In [1]:
import pandas as pd
import numpy as np

In [2]:
ames = pd.read_csv('ames_prices.csv')
nhanes = pd.read_csv('nhanes_data_17_18.csv')
metabric = pd.read_csv('metabric.csv')

C:\Users\micha\AppData\Local\Temp\ipykernel_25348\2489820013.py:2: DtypeWarning: Columns (142) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv('nhanes_data_17_18.csv')


In [3]:
ames.columns

Index(['Order', 'PID', 'area', 'price', 'MS.SubClass', 'MS.Zoning',
       'Lot.Frontage', 'Lot.Area', 'Street', 'Alley', 'Lot.Shape',
       'Land.Contour', 'Utilities', 'Lot.Config', 'Land.Slope', 'Neighborhood',
       'Condition.1', 'Condition.2', 'Bldg.Type', 'House.Style',
       'Overall.Qual', 'Overall.Cond', 'Year.Built', 'Year.Remod.Add',
       'Roof.Style', 'Roof.Matl', 'Exterior.1st', 'Exterior.2nd',
       'Mas.Vnr.Type', 'Mas.Vnr.Area', 'Exter.Qual', 'Exter.Cond',
       'Foundation', 'Bsmt.Qual', 'Bsmt.Cond', 'Bsmt.Exposure',
       'BsmtFin.Type.1', 'BsmtFin.SF.1', 'BsmtFin.Type.2', 'BsmtFin.SF.2',
       'Bsmt.Unf.SF', 'Total.Bsmt.SF', 'Heating', 'Heating.QC', 'Central.Air',
       'Electrical', 'X1st.Flr.SF', 'X2nd.Flr.SF', 'Low.Qual.Fin.SF',
       'Bsmt.Full.Bath', 'Bsmt.Half.Bath', 'Full.Bath', 'Half.Bath',
       'Bedroom.AbvGr', 'Kitchen.AbvGr', 'Kitchen.Qual', 'TotRms.AbvGrd',
       'Functional', 'Fireplaces', 'Fireplace.Qu', 'Garage.Type',
       'Garage.Yr.B

# 7

a)

In [4]:
ames['Central.Air.Indicator'] = (ames['Central.Air'] == "Y").astype(int)
estimate = ames['Central.Air.Indicator'].mean()
print(estimate)

0.9331058020477816


The proportion of homes that have central air conditioning is 2734/2930

In [5]:
#b)

np.random.seed(549)

bootstrap_ests = []

for i in range(2000):
    bootstrap_sample = ames['Central.Air.Indicator'].sample(
        frac = 1, replace = True
    )
    bootstrap_ests.append(bootstrap_sample.mean())
bootstrap_ests[:10]

[np.float64(0.9358361774744027),
 np.float64(0.9208191126279863),
 np.float64(0.931740614334471),
 np.float64(0.9320819112627986),
 np.float64(0.9378839590443686),
 np.float64(0.9245733788395905),
 np.float64(0.9402730375426621),
 np.float64(0.9273037542662116),
 np.float64(0.9303754266211605),
 np.float64(0.9344709897610921)]

In [6]:
#c

bootstrap_se = np.std(bootstrap_ests, ddof=1)
print(bootstrap_se)

0.004684692344853809


In [7]:
#d
from scipy.stats import norm

se = bootstrap_se

for confidence in [0.90, 0.95, 0.99]:
    z = norm.ppf(1 - (1 - confidence) / 2)
    lower = estimate - z * se
    upper = estimate + z * se

    print(f"{confidence:.0%} CI: ({lower:.4f}, {upper:.4f})")

90% CI: (0.9254, 0.9408)
95% CI: (0.9239, 0.9423)
99% CI: (0.9210, 0.9452)


e)

.9 does not fall into any of the confidence intervals above, so we reject the null hypothesis that p = 0.9. Based on this, there is only a .01 probability that the proportion of houses with central air conditioning isn't 0.9 just by random chance. It is thus highly likely that the true proportion is not .9 and instead lies in the interval (.921, .9452)

# 8

a)

In [8]:
np.corrcoef(
    np.arcsinh(ames["area"]),
    np.arcsinh(ames["price"])
)[0, 1]

np.float64(0.7231661530091356)

In [9]:
np.random.seed(929)

bootstrap_ests2 = []

for i in range(3000):
    bootstrap_sample = ames.sample(frac=1, replace=True)

    bootstrap_sample_cors = np.corrcoef(
        np.arcsinh(bootstrap_sample["area"]),
        np.arcsinh(bootstrap_sample["price"])
    )[0, 1]

    bootstrap_ests2.append(bootstrap_sample_cors)

bootstrap_ests2[:10]


[np.float64(0.7188912066673299),
 np.float64(0.7276540731846572),
 np.float64(0.7184932772064748),
 np.float64(0.7294816842718514),
 np.float64(0.7198861473544518),
 np.float64(0.7144802770201527),
 np.float64(0.7395901974682486),
 np.float64(0.7254546876609315),
 np.float64(0.7198344765113983),
 np.float64(0.7298110002297565)]

There were 3000 bootstrap replications and the seed was 929

In [10]:
#b

bootstrap_se2 = np.std(bootstrap_ests2, ddof=1)
print(bootstrap_se2)

0.01033417354285633


In [11]:
#c

estimate2 = np.corrcoef(
    np.arcsinh(ames["area"]),
    np.arcsinh(ames["price"])
)[0, 1]

from scipy.stats import norm

for confidence in [0.90, 0.95, 0.99]:
    z = norm.ppf(1 - (1 - confidence) / 2)
    lower = estimate2 - z * bootstrap_se2
    upper = estimate2 + z * bootstrap_se2

    print(f"{confidence:.0%} CI: ({lower:.4f}, {upper:.4f})")

90% CI: (0.7062, 0.7402)
95% CI: (0.7029, 0.7434)
99% CI: (0.6965, 0.7498)


d)

Because 0 isn't within any of the confidence intervals above, we can reject the null hypothesis and conclude that the correlation between area and price is not 0. 

e)

When bootstrapping a statistic that measures a relationship between two variables, it's importatnt to resample both variables together. Each home's area and price belong together as one observation. When estimating their Pearson correlation, that relationship needs to be preserved during resampling.

# 9

a)

In [16]:
metabric_estimate = metabric['Overall Survival (Months)'].mean()
print(metabric_estimate)

129.09429138744105


In [39]:
np.random.seed(145)

bootstrap_ests3 = []

for i in range(3000):
    bootstrap_sample = metabric.sample(frac=1, replace=True)

    bootstrap_sample_means = bootstrap_sample['Overall Survival (Months)'].mean()

    bootstrap_ests3.append(bootstrap_sample_means)

bootstrap_ests3[:10]

[np.float64(132.40729709605358),
 np.float64(131.20049640109207),
 np.float64(131.56219905683793),
 np.float64(128.96738644825018),
 np.float64(126.23936460660217),
 np.float64(129.160312732688),
 np.float64(129.9656738644825),
 np.float64(130.85450483991062),
 np.float64(123.53668404070488),
 np.float64(130.85443037974684)]

There were 3000 replications and the seed was 145. 

b)

In [40]:
bootstrap_se3 = np.std(bootstrap_ests3, ddof=1)
print(bootstrap_se3)

2.083497778732277


In [41]:
#c)


for confidence in [0.90, 0.95, 0.99]:
    z = norm.ppf(1 - (1 - confidence) / 2)
    lower = metabric_estimate - z * bootstrap_se3
    upper = metabric_estimate + z * bootstrap_se3

    print(f"{confidence:.0%} CI: ({lower:.4f}, {upper:.4f})")


90% CI: (125.6672, 132.5213)
95% CI: (125.0107, 133.1779)
99% CI: (123.7276, 134.4610)


d)

We can reject the null hypothesis that the average overall survival in months is 120. This is because 120 does not lie in any of the intervals from the three confidence intervals. 

e)

It is highly likely the true overall survival rate in months lies roughly between 123 and 134 based on our confidence intervals. This corresponds to a survival rate in years of roughly 10 years. This likely does not estimate unrestricted mean lifetime because many of the observations that contribute to the estimate are of patients who are still living. Moreover, the overall survival depends heavily on the exact type and stage of cancer, with the lifetimes varying extensively. 

# 10

a)

In [33]:
nhanes['SleepHoursWeekdaysOrWorkdays'].head(15)

0      NaN
1      NaN
2      8.0
3     10.5
4      NaN
5      8.0
6      7.0
7      7.0
8      7.5
9      5.5
10     7.0
11     5.0
12     7.0
13     8.0
14     8.0
Name: SleepHoursWeekdaysOrWorkdays, dtype: float64

In [35]:
nhanes_18 = nhanes[
    (nhanes["AgeInYearsAtScreening"] >= 18) &
    (nhanes["SleepHoursWeekdaysOrWorkdays"].notna())
]

sleep_est = nhanes_18['SleepHoursWeekdaysOrWorkdays'].mean()
print(sleep_est)


7.62124248496994


In [38]:
np.random.seed(81)

sleep_boostrap_ests = []
for i in range(2000):
    bootstrap_sample = nhanes_18.sample(frac = 1, replace = True)
    bootstrap_sample_sleepmeans = bootstrap_sample['SleepHoursWeekdaysOrWorkdays'].mean()
    sleep_boostrap_ests.append(bootstrap_sample_sleepmeans)

sleep_boostrap_ests[:10]

[np.float64(7.622062306431044),
 np.float64(7.587174348697395),
 np.float64(7.6198761158681),
 np.float64(7.637547822918564),
 np.float64(7.636181453816724),
 np.float64(7.606121333576243),
 np.float64(7.634359628347604),
 np.float64(7.655948260156677),
 np.float64(7.598196392785571),
 np.float64(7.633448715613044)]

In [44]:
#b
bootstrap_std_sleep = np.std(sleep_boostrap_ests, ddof = 1)
print(bootstrap_std_sleep)

0.022783203363389005


In [45]:
#c


for confidence in [0.90, 0.95, 0.99]:
    z = norm.ppf(1 - (1 - confidence) / 2)
    lower = sleep_est - z * bootstrap_std_sleep
    upper = sleep_est + z * bootstrap_std_sleep

    print(f"{confidence:.0%} CI: ({lower:.4f}, {upper:.4f})")

90% CI: (7.5838, 7.6587)
95% CI: (7.5766, 7.6659)
99% CI: (7.5626, 7.6799)


d)

Based on the three confidence intervals above, we can reject the null hypothesis that the true weekday mean hours of sleep is 8. This is because 8 does not lie within any of the confidence intervals above.

e)

All three confidence intervals are quite rigid, indicating a very consistent mean weekday hours of sleep across the dataset. It is highly likely that the true mean is roughly 7.6 hours. Practically, this is somewhat plausible, but for the general population I would imagine this varies sigificantly based on the industry the respondant worked in. It is fairly likely that the average adult gets roughly 7.5 hours of sleep on a weekday, however I imagine for some industries this number is far lower. However, based on human biology, it is very unlikely that any concentration of people is averaging significantly higher hours of sleep on weekdays. I would imagine the most likely upper end to be about 9 hours for any particular group of people. 